# 07 — Production Router: Examples

> Combine routing, fallback, circuit breakers, and metrics into a single production-grade class.

---

**What you'll build:**
1. Full `ProductionRouter` class with all patterns integrated
2. Rule engine wired to model groups
3. Live tests with metrics
4. Chaos test — simulated failures

In [ ]:
# !pip install openai tenacity python-dotenv rich tiktoken

In [ ]:
import os, time, random, json, hashlib, logging
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from enum import Enum
from typing import Callable
from collections import defaultdict
import tiktoken

from openai import OpenAI, RateLimitError, APIStatusError, APITimeoutError
from dotenv import load_dotenv
from rich import print as rprint
from rich.table import Table
from rich.console import Console

load_dotenv()
logging.basicConfig(level=logging.WARNING)
logger = logging.getLogger(__name__)

console = Console()
enc = tiktoken.encoding_for_model("gpt-4o-mini")
base_client = OpenAI()
print('✅ Setup complete')

---
## Part 1: Core Data Structures

In [ ]:
@dataclass
class RouterResponse:
    content: str
    model_used: str
    provider: str
    input_tokens: int
    output_tokens: int
    latency_ms: float
    fallback_used: bool
    attempt_count: int
    cost_usd: float
    routing_reason: str
    from_cache: bool = False


@dataclass
class RoutingRule:
    name: str
    condition: Callable[[dict], bool]
    model_group: str
    priority: int
    reason: str


class CircuitState(Enum):
    CLOSED    = "CLOSED"
    OPEN      = "OPEN"
    HALF_OPEN = "HALF_OPEN"


@dataclass
class CircuitBreaker:
    name: str
    failure_threshold: int = 5
    recovery_timeout: int = 60
    state: CircuitState = field(default=CircuitState.CLOSED, init=False)
    failure_count: int = field(default=0, init=False)
    last_failure: datetime | None = field(default=None, init=False)

    def can_proceed(self) -> bool:
        if self.state == CircuitState.OPEN:
            if self.last_failure and datetime.now() > self.last_failure + timedelta(seconds=self.recovery_timeout):
                self.state = CircuitState.HALF_OPEN
                return True
            return False
        return True

    def success(self):
        self.failure_count = 0
        if self.state != CircuitState.CLOSED:
            rprint(f"   ✅ Circuit CLOSED for [green]{self.name}[/green]")
        self.state = CircuitState.CLOSED

    def failure(self):
        self.failure_count += 1
        self.last_failure = datetime.now()
        if self.failure_count >= self.failure_threshold:
            self.state = CircuitState.OPEN
            rprint(f"   🔴 Circuit OPEN for [red]{self.name}[/red] ({self.failure_count} failures)")

    def status(self) -> str:
        icons = {CircuitState.CLOSED: "✅", CircuitState.OPEN: "🔴", CircuitState.HALF_OPEN: "🟡"}
        return f"{icons[self.state]} {self.state.value}"


print('✅ Data structures defined')

---
## Part 2: ProductionRouter Class

In [ ]:
class ProductionRouter:
    """
    Production-grade LLM router.
    Integrates:
    - Priority-ordered rule engine for model group selection
    - Multi-model fallback chain per group
    - Per-model circuit breakers
    - Exponential backoff retry for rate limits
    - Response caching as last resort
    - Structured metrics logging
    """

    PRICING = {
        "gpt-4o":                    (5.00e-6, 15.00e-6),
        "gpt-4o-mini":               (0.15e-6,  0.60e-6),
        "claude-3-5-sonnet-20241022":(3.00e-6, 15.00e-6),
        "claude-3-5-haiku-20241022": (0.80e-6,  4.00e-6),
        "gemini-1.5-flash":          (0.075e-6, 0.30e-6),
        "gemini-1.5-pro":            (1.25e-6,  5.00e-6),
    }

    def __init__(self, model_groups: dict[str, list[str]], default_group: str = "standard"):
        self.model_groups = model_groups
        self.default_group = default_group
        self.rules: list[RoutingRule] = []
        self.breakers: dict[str, CircuitBreaker] = {
            model: CircuitBreaker(name=model)
            for models in model_groups.values()
            for model in models
        }
        self._cache: dict[str, str] = {}
        self._metrics: list[dict] = []

    def add_rule(self, rule: RoutingRule) -> "ProductionRouter":
        self.rules.append(rule)
        self.rules.sort(key=lambda r: r.priority)
        return self

    def _select_group(self, ctx: dict) -> tuple[str, str]:
        for rule in self.rules:
            try:
                if rule.condition(ctx):
                    return rule.model_group, f"Rule '{rule.name}': {rule.reason}"
            except Exception:
                pass
        return self.default_group, "Default group"

    def _call_model(self, model: str, messages: list[dict], **kw) -> dict:
        """Single model call. Uses gpt-4o-mini as stand-in for non-OpenAI models."""
        call_model = model if model.startswith("gpt") else "gpt-4o-mini"
        resp = base_client.chat.completions.create(
            model=call_model, messages=messages, max_tokens=kw.get("max_tokens", 300)
        )
        return {
            "content": resp.choices[0].message.content,
            "input_tokens": resp.usage.prompt_tokens,
            "output_tokens": resp.usage.completion_tokens,
        }

    def _cost(self, model: str, in_tok: int, out_tok: int) -> float:
        pi, po = self.PRICING.get(model, (5e-6, 15e-6))
        return pi * in_tok + po * out_tok

    def _cache_key(self, messages: list[dict]) -> str:
        return hashlib.md5(json.dumps(messages, sort_keys=True).encode()).hexdigest()

    def complete(self, messages: list[dict], context: dict | None = None, **kw) -> RouterResponse:
        start = time.time()
        ctx = context or {}
        cache_key = self._cache_key(messages)

        group, reason = self._select_group(ctx)
        chain = self.model_groups.get(group, self.model_groups[self.default_group])

        attempt = 0
        last_error = None

        for i, model in enumerate(chain):
            breaker = self.breakers[model]
            if not breaker.can_proceed():
                rprint(f"   ⚡ [{model}] circuit open — skipping")
                continue

            for retry in range(3):
                attempt += 1
                try:
                    result = self._call_model(model, messages, **kw)
                    breaker.success()

                    latency = (time.time() - start) * 1000
                    cost = self._cost(model, result["input_tokens"], result["output_tokens"])
                    self._cache[cache_key] = result["content"]
                    self._metrics.append({
                        "ts": datetime.now().isoformat(), "model": model,
                        "latency_ms": round(latency, 1), "cost_usd": cost,
                        "fallback_used": i > 0, "attempts": attempt, "reason": reason,
                    })

                    return RouterResponse(
                        content=result["content"], model_used=model,
                        provider="openai" if "gpt" in model else "other",
                        input_tokens=result["input_tokens"],
                        output_tokens=result["output_tokens"],
                        latency_ms=round(latency, 1), fallback_used=(i > 0),
                        attempt_count=attempt, cost_usd=cost, routing_reason=reason,
                    )

                except RateLimitError:
                    if retry < 2:
                        delay = min(1.0 * (2**retry) + random.uniform(0, 0.5), 15)
                        rprint(f"   ⚠️ [{model}] Rate limit. Retry {retry+1} in {delay:.1f}s...")
                        time.sleep(delay)
                    else:
                        breaker.failure(); last_error = "rate_limit"; break

                except Exception as e:
                    breaker.failure(); last_error = str(e); break

        # Cache fallback
        if (cached := self._cache.get(cache_key)):
            rprint("   📦 [yellow]Returning cached response[/yellow]")
            return RouterResponse(
                content=cached, model_used="cache", provider="cache",
                input_tokens=0, output_tokens=0,
                latency_ms=round((time.time()-start)*1000, 1),
                fallback_used=True, attempt_count=attempt, cost_usd=0,
                routing_reason=reason, from_cache=True,
            )

        raise Exception(f"All models failed. Last error: {last_error}")

    def metrics_summary(self):
        if not self._metrics:
            print("No requests logged.")
            return
        total_cost = sum(m["cost_usd"] for m in self._metrics)
        avg_lat = sum(m["latency_ms"] for m in self._metrics) / len(self._metrics)
        fallbacks = sum(1 for m in self._metrics if m["fallback_used"])

        table = Table(title="📊 Router Metrics Summary", show_header=True)
        table.add_column("Metric", style="cyan")
        table.add_column("Value", style="green")
        table.add_row("Total requests", str(len(self._metrics)))
        table.add_row("Total cost", f"${total_cost:.5f}")
        table.add_row("Avg latency", f"{avg_lat:.0f}ms")
        table.add_row("Fallback rate", f"{fallbacks}/{len(self._metrics)} ({fallbacks/len(self._metrics)*100:.0f}%)")
        table.add_row("Circuit states", ", ".join(f"{m}: {b.status()}" for m, b in self.breakers.items()))
        console.print(table)


print("✅ ProductionRouter defined")

---
## Part 3: Wire Up Rules & Run

In [ ]:
# Build the router
router = ProductionRouter(
    model_groups={
        "premium":  ["gpt-4o", "claude-3-5-sonnet-20241022"],
        "standard": ["gpt-4o-mini", "claude-3-5-haiku-20241022"],
        "fast":     ["gemini-1.5-flash", "gpt-4o-mini"],
    },
    default_group="standard"
)

# Add routing rules
(
    router
    .add_rule(RoutingRule(
        name="long_context",
        condition=lambda ctx: ctx.get("input_tokens", 0) > 64_000,
        model_group="premium", priority=1,
        reason="Large input → premium model"
    ))
    .add_rule(RoutingRule(
        name="realtime",
        condition=lambda ctx: ctx.get("max_latency_ms", float("inf")) < 1000,
        model_group="fast", priority=2,
        reason="Latency < 1s → fast model"
    ))
    .add_rule(RoutingRule(
        name="high_stakes",
        condition=lambda ctx: ctx.get("task") in {"legal", "medical", "contract"},
        model_group="premium", priority=3,
        reason="High-stakes task → premium model"
    ))
    .add_rule(RoutingRule(
        name="budget",
        condition=lambda ctx: ctx.get("max_cost_usd", float("inf")) < 0.005,
        model_group="standard", priority=4,
        reason="Budget constraint → standard model"
    ))
)

print(f"✅ Router configured with {len(router.rules)} rules and {len(router.breakers)} circuit breakers")

In [ ]:
# ── Run test requests ──────────────────────────────────────────────────────
test_requests = [
    {
        "messages": [{"role": "user", "content": "Explain what circuit breakers are in software."}],
        "context": {},
        "label": "Standard request"
    },
    {
        "messages": [{"role": "user", "content": "Translate: 'The meeting is at 3pm'"}],
        "context": {"max_latency_ms": 800},
        "label": "Real-time (< 1s)"
    },
    {
        "messages": [{"role": "user", "content": "Review this software license clause for risks."}],
        "context": {"task": "legal"},
        "label": "Legal task"
    },
    {
        "messages": [{"role": "user", "content": "Summarize what exponential backoff does."}],
        "context": {"max_cost_usd": 0.002},
        "label": "Budget constrained"
    },
]

table = Table(title="ProductionRouter Test Results", show_header=True)
table.add_column("Label", style="cyan")
table.add_column("Model", style="green")
table.add_column("Group Match", style="yellow")
table.add_column("Latency", style="white")
table.add_column("Cost", style="white")
table.add_column("Fallback?", style="red")

for req in test_requests:
    resp = router.complete(
        messages=req["messages"],
        context=req["context"],
        max_tokens=100
    )
    table.add_row(
        req["label"],
        resp.model_used,
        resp.routing_reason[:40] + ".." if len(resp.routing_reason) > 40 else resp.routing_reason,
        f"{resp.latency_ms}ms",
        f"${resp.cost_usd:.6f}",
        "Yes" if resp.fallback_used else "No"
    )

console.print(table)

In [ ]:
# ── Print metrics ──────────────────────────────────────────────────────────
router.metrics_summary()

---
## Part 4: Chaos Test — Injected Failures

In [ ]:
# ── Simulate circuit breaker opening via repeated failures ─────────────────

# Manually trigger failures on the first model in 'standard' group
failing_model = "gpt-4o-mini"
breaker = router.breakers[failing_model]

print(f"🧪 Chaos test: injecting {breaker.failure_threshold} failures into [{failing_model}]\n")
for i in range(breaker.failure_threshold):
    breaker.failure()
    print(f"   Failure {i+1}: circuit state = {breaker.status()}")

print(f"\n   Circuit state for {failing_model}: {breaker.status()}")
print(f"   can_proceed: {breaker.can_proceed()}")

print("\n📤 Sending request — router should skip failing model and use fallback...\n")

# This should skip gpt-4o-mini and try the next in chain
resp = router.complete(
    messages=[{"role": "user", "content": "What is a circuit breaker pattern?"}],
    max_tokens=80
)

rprint(f"\n✅ [bold green]Response received despite injected failures![/bold green]")
rprint(f"   Model used:    {resp.model_used}")
rprint(f"   Fallback used: {resp.fallback_used}")
rprint(f"   From cache:    {resp.from_cache}")
rprint(f"   Content:       {resp.content[:150]}")

In [ ]:
# ── Recovery: reset the circuit breaker ───────────────────────────────────
print("🔧 Manually recovering circuit breaker (simulating provider returning to health)...")
breaker.success()
print(f"   {failing_model}: {breaker.status()} | can_proceed: {breaker.can_proceed()}")

# Now route normally again
resp = router.complete(
    messages=[{"role": "user", "content": "What is exponential backoff?"}],
    max_tokens=60
)
rprint(f"\n✅ Normal routing resumed — model: {resp.model_used}")

In [ ]:
# ── Final metrics after chaos test ────────────────────────────────────────
router.metrics_summary()

---
## 🎉 Module Complete!

You've now built every layer of a production LLM routing system:

| Topic | Achievement |
|---|---|
| `01_what_is_routing` | Understand routing, fallback, load balancing |
| `02_rule_based_routing` | Token, keyword, cost, latency rules |
| `03_semantic_routing` | Embedding-based intent routing |
| `04_fallback_strategies` | Retry, circuit breaker, cache fallback |
| `05_load_balancing` | Round-robin, weighted, rate-limit-aware |
| `06_litellm_router` | Production router with zero boilerplate |
| `07_production_router` | Full hardened router class with monitoring |